# 🎓 Lesson 04 — Build Your First AI Drafter

**The seed of the whole course. By the end of this notebook you have a working drafter that reads a customer email, finds the right shipment, and writes a reply in PacificFreight's voice.**

Time: 30 min to read + 30 min to run. Topic: the 5-piece anatomy of every FDE tool.

---

## 🎯 What you will build

```
  📧 Customer email: "Where is my parcel PF-1003?"
                │
                ▼
  ┌──────────────────────────────────┐
  │  🤖 The 5-piece drafter          │
  │                                  │
  │  1️⃣  READ    the email           │
  │  2️⃣  EXTRACT the shipment ID     │
  │  3️⃣  LOOKUP  the shipment data   │
  │  4️⃣  DRAFT   a reply with the LLM │
  │  5️⃣  OUTPUT  the draft to Mei    │
  └──────────────────────────────────┘
                │
                ▼
  ✉️  Draft: "Hi! Your parcel PF-1003 is in customs at HCMC."
```

**The big idea:** every AI tool is just 5 small steps glued together. Once you see the 5 pieces, you can build ANY AI tool. The pieces are like LEGO bricks — they always snap together the same way.

## 🧠 Concept (5 min) — What is an LLM, really?

Before we build anything, let's understand the magic box.

An **LLM** (Large Language Model) is a program that:

1. Has read **millions of pages** of text from the internet, books, and code
2. Learned the **patterns** in that text (which words follow which other words)
3. When you give it a sentence, it **predicts the next word**, then the next, then the next

That's it. It's an **autocomplete on steroids.**

```
  You type:  "The cat sat on the"
                │
                ▼
  LLM thinks:  "mat"  (most common next word)
                │
                ▼
  You see:    "The cat sat on the mat"

  The LLM is just a really good guesser. It guesses the next word.
  Then it guesses the next word after that.
  Then the next.  And the next.  And the next.
```

**Why is this useful for customer service?** Because Mei writes emails all day that look like:

```
  "Dear customer, your parcel PF-1003 is currently in customs at HCMC.
   It will be released within 24 hours. Thank you for your patience.
   Best regards, Mei from PacificFreight"
```

The LLM has read millions of business emails. It can write a *new* email in that same style — without ever seeing Mei's exact email. **That's the magic.**

**But wait — there's a catch.** The LLM doesn't know about PF-1003 specifically. It doesn't know PF-1003 is in customs. **It only knows what we tell it.** That's why we have the 5-piece drafter: the 4 OTHER pieces (READ, EXTRACT, LOOKUP, OUTPUT) gather the *facts*, and the LLM (DRAFT) writes the *words*.

## 🏗️ Build it, step by step

We'll build the 5 pieces in order. After each piece, we'll test it. By the end, the pieces snap together into a working drafter.

### Step 0 — Set up

First, let's import the libraries we need and define our data.

Think of `import` like opening a toolbox. We need a few tools: `re` for finding patterns in text, `json` for reading data files, and `Path` for finding files on the computer.

In [ ]:
# Step 0 — imports (open the toolbox)
import json
import re
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

# Find phase-1-foundations/shared/ by searching ancestors and common subpaths
PHASE1 = None
search_paths = []
for base in [Path.cwd(), *Path.cwd().parents]:
    search_paths.extend([
        base / "shared" / "shipments.json",
        base / "phase-1-foundations" / "shared" / "shipments.json",
        base / "course" / "ai-fde" / "phase-1-foundations" / "shared" / "shipments.json",
    ])
for p in search_paths:
    if p.exists():
        PHASE1 = p.parent.parent  # the phase-1-foundations/ dir
        break

if PHASE1 is None:
    raise FileNotFoundError("Could not find phase-1-foundations/shared/shipments.json anywhere")

print(f"Data from: {PHASE1}")
print(f"  Shipments: {(PHASE1 / 'shared' / 'shipments.json').exists()}")
print(f"  Style guide: {(PHASE1 / 'shared' / 'style-guide.md').exists()}")
print(f"  Sample emails: {(PHASE1 / 'shared' / 'sample-emails.md').exists()}")


### Step 1 — READ the email

The first piece of every AI tool is **READ**: get the customer's question into the computer.

Why is this a separate step? Because the input can come from many places:
- A file (we'll read this way today)
- A copy-paste from Gmail
- An HTTP request from a website (Phase 2!)
- A webhook from a help-desk system

By keeping READ separate, we can swap inputs without changing the rest of the tool.

In [ ]:
# Step 1 — READ the email
def read_email(source: str) -> str:
    """Read an email from a file path or return the string directly.

    This is the READ step. It takes a source (file path or string) and
    returns the email text. In Phase 2 this becomes an HTTP POST body.
    """
    p = Path(source)
    if p.exists() and p.is_file():
        return p.read_text(encoding="utf-8")
    return source  # treat as inline text

# Test it!
email = read_email(str(PHASE1 / "shared" / "sample-emails.md"))
print(f"📧 Read {len(email)} characters of email content.")
print(f"   First 200 chars: {email[:200]!r}")

### Step 2 — EXTRACT the shipment ID

Most customer emails mention a shipment ID like `PF-1003`. We need to find it.

**Why use a regex instead of asking the LLM?**

| Method | Speed | Cost | Accuracy |
|---|---|---|---|
| Regex (pattern match) | ⚡ 0.01ms | $0 | 95% on clean emails |
| LLM (ask the AI) | 🐢 800ms | $0.001 | 99% on messy emails |

The FDE move: use the **cheap tool first**, the **expensive tool as fallback**. Try regex; if regex fails, ask the LLM. This is the same idea as "tie your shoes before you buy new shoes."

**The regex explained:**
```
PF-\d{4}
│   │  │
│   │  └─ exactly 4 digits
│   └───── a digit character (0-9)
│         (the {4} says "4 of them")
└───────── the literal characters "PF-"
```
Translation: "find the letters P-F-dash followed by exactly 4 digits."

In [ ]:
# Step 2 — EXTRACT the shipment ID
_SHIPMENT_ID_REGEX = re.compile(r"PF-\d{4}")

def extract_shipment_id(email_text: str) -> Optional[str]:
    """Find a PF-XXXX shipment ID in the email. Returns None if not found.

    Try regex first (cheap). If that fails, the caller can fall back to
    asking the LLM to extract it (expensive). We do that fallback in
    Step 4 (DRAFT).
    """
    match = _SHIPMENT_ID_REGEX.search(email_text)
    return match.group(0) if match else None

# Test it on a real email
test_email = "Hi! I sent parcel PF-1003 last Tuesday and haven't heard back."
shipment_id = extract_shipment_id(test_email)
print(f"🔍 Found: {shipment_id}")
assert shipment_id == "PF-1003", "Regex should find PF-1003"

# Test it on a messy email
messy = "hey, where is my thing?? the one i sent on the 12th, the number was PF-1007 or something"
print(f"🔍 Messy: {extract_shipment_id(messy)}")  # still finds PF-1007

# Test it on an email with no ID
no_id = "I want to know if you ship to Vietnam?"
print(f"🔍 No ID: {extract_shipment_id(no_id)}")  # returns None

### Step 3 — LOOKUP the shipment data

Now we have a shipment ID. We need to find out where that parcel is.

**Why a separate LOOKUP step?** Because the data can come from many places too:
- A local JSON file (today — 15 mock shipments)
- A real database (Postgres, in production)
- An API call to the courier (DHL, FedEx)
- A spreadsheet Sarah maintains in Google Sheets

By keeping LOOKUP separate, we can swap data sources without changing READ or DRAFT. **The pieces are LEGO bricks — they snap together, and you can swap any one without breaking the others.**

**The data structure explained:**
```json
{
  "id": "PF-1003",
  "status": "in_customs",
  "origin": "Singapore",
  "destination": "Ho Chi Minh City",
  "last_event": "Arrived at HCMC customs, 2026-10-05 14:23",
  "customer_name": "Linh Nguyen"
}
```
Each field is a *fact* about the shipment. The LLM will use these facts to write a natural-sounding reply.

In [ ]:
# Step 3 — LOOKUP the shipment data
@dataclass(frozen=True)
class Shipment:
    """A parcel being shipped. Frozen means the fields can't change after creation.

    We use a dataclass (a fancy dict) so the code is type-safe and self-documenting.
    When you see Shipment, you know exactly which fields it has.
    """
    id: str
    status: str
    origin: str
    destination: str
    last_event: str          # human-readable "what happened" string
    customer_name: str


def load_shipments(path: Path) -> dict[str, Shipment]:
    """Load the shipments from a JSON file. Returns a dict {id: Shipment}.

    The shipments.json file is wrapped in a metadata envelope:
        {"schema_version": "1.0", "as_of": "...", "shipments": [ {...}, ... ]}
    We unwrap the envelope and build a {id: Shipment} dict. Any extra fields
    in the JSON (customer_email, last_event_at, events[]) are ignored.
    """
    raw = json.loads(path.read_text(encoding="utf-8"))
    # Handle both the wrapped shape {"shipments": [...]} and a bare list [...]
    items = raw["shipments"] if isinstance(raw, dict) and "shipments" in raw else raw
    return {
        s["id"]: Shipment(
            id=s["id"],
            status=s["status"],
            origin=s["origin"],
            destination=s["destination"],
            last_event=s["last_event"],
            customer_name=s["customer_name"],
        )
        for s in items
    }

def lookup_shipment(shipments: dict[str, Shipment], shipment_id: str) -> Optional[Shipment]:
    """Look up a shipment by ID. Returns None if not found."""
    return shipments.get(shipment_id)


# Test it
shipments = load_shipments(PHASE1 / "shared" / "shipments.json")
print(f"📦 Loaded {len(shipments)} shipments.")
print(f"   PF-1003: {lookup_shipment(shipments, 'PF-1003')}")
print(f"   PF-9999 (doesn't exist): {lookup_shipment(shipments, 'PF-9999')}")

### Step 4 — DRAFT the reply (the magic step)

Now we have:
- The customer's email
- The shipment ID
- The shipment's data (status, origin, destination, last event)

Time to ask the LLM to write a reply. But we don't just say "write a reply." We give the LLM a **structured prompt** with three parts:

```
  ┌─────────────────────────────────────────────────────┐
  │  📜 SYSTEM PROMPT (who you are)                    │
  │                                                     │
  │  "You are Mei, a customer-service rep at            │
  │   PacificFreight. You write short, warm replies.    │
  │   Never promise specific dates you don't know."    │
  │                                                     │
  │  Why: tells the LLM the persona, the style,        │
  │  and the rules (don't make up dates).              │
  └─────────────────────────────────────────────────────┘
  ┌─────────────────────────────────────────────────────┐
  │  📚 CONTEXT (the facts)                            │
  │                                                     │
  │  "Shipment PF-1003 is in customs at HCMC.           │
  │   Last event: Arrived at HCMC customs, 2026-10-05. │
  │   Customer: Linh Nguyen."                          │
  │                                                     │
  │  Why: gives the LLM the FACTS so it doesn't       │
  │  make them up. The LLM is a writer, not a          │
  │  database. We provide the database.                │
  └─────────────────────────────────────────────────────┘
  ┌─────────────────────────────────────────────────────┐
  │  ❓ USER MESSAGE (the question)                    │
  │                                                     │
  │  "Customer email: 'Where is my parcel PF-1003?'"     │
  │                                                     │
  │  Why: this is what the LLM is responding TO.        │
  └─────────────────────────────────────────────────────┘
```

**The 3-part prompt is the secret sauce.** Without the system prompt, the LLM might reply in a pirate voice. Without the context, the LLM might make up facts. Without the user message, the LLM doesn't know what to write about. **All 3 are needed.**

In [ ]:
# Step 4 — DRAFT the reply
def build_prompt(email_text: str, shipment: Shipment, style_guide: str) -> list[dict]:
    """Build the 3-part prompt: system + context + user.

    Returns a list of message dicts in OpenAI's format:
      [{"role": "system", "content": ...}, {"role": "user", "content": ...}]
    """
    system = (
        "You are Mei, a customer-service representative at PacificFreight Co., "
        "a 12-person cross-border logistics company shipping between Singapore and Vietnam.\n\n"
        "TONE RULES:\n"
        "- Warm but professional. Use the customer's first name.\n"
        "- Short. 2-4 sentences max.\n"
        "- Never invent dates, prices, or policies not given to you.\n"
        "- If you don't know something, say so and offer to escalate.\n\n"
        "STYLE GUIDE (excerpts from PacificFreight's handbook):\n"
        f"{style_guide}"
    )
    context = (
        f"FACTS about the shipment:\n"
        f"  ID: {shipment.id}\n"
        f"  Status: {shipment.status}\n"
        f"  Origin: {shipment.origin}\n"
        f"  Destination: {shipment.destination}\n"
        f"  Last event: {shipment.last_event}\n"
        f"  Customer name: {shipment.customer_name}\n"
    )
    user = f"Customer email:\n{email_text}\n\nWrite a reply. Sign as Mei."
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": f"{context}\n\n{user}"},
    ]

# Now define the LLM call. We use a "mock" so it works without an API key.
def complete(messages: list[dict], model: str = "mock") -> str:
    """Send messages to the LLM. Returns the assistant's reply as a string.

    The 'mock' model is a fake LLM that returns a deterministic response
    based on the input. This means the drafter ALWAYS works, even
    without an OpenAI API key. When you have a key, swap in real OpenAI.
    """
    if model == "mock" or not _has_openai():
        return _mock_complete(messages)
    return _openai_complete(messages, model)

def _has_openai() -> bool:
    import os
    return bool(os.environ.get("OPENAI_API_KEY"))

def _mock_complete(messages: list[dict]) -> str:
    """A fake LLM that picks up the customer name + status from the prompt.

    The mock is 'good enough' for testing the pipeline. It reads the
    structured prompt and returns a template response. When you have
    a real API key, the real LLM will write much better replies.
    """
    text = " ".join(m["content"] for m in messages)
    # Pull the customer name
    import re
    name_match = re.search(r"Customer name: ([\w ]+)", text)
    name = name_match.group(1).strip() if name_match else "there"
    # Pull the status
    status_match = re.search(r"Status: ([\w_]+)", text)
    status = status_match.group(1).strip() if status_match else "in transit"
    return (
        f"Hi {name},\n\n"
        f"Thanks for reaching out. Your shipment is currently {status.replace('_', ' ')}. "
        f"I'll keep an eye on it and let you know the moment it moves.\n\n"
        f"Best,\nMei"
    )

def _openai_complete(messages: list[dict], model: str) -> str:
    """Real OpenAI call. Requires OPENAI_API_KEY in your environment."""
    import os
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    resp = client.chat.completions.create(model=model, messages=messages)
    return resp.choices[0].message.content


# Test it on PF-1003
style_guide = (PHASE1 / "shared" / "style-guide.md").read_text(encoding="utf-8")
shipment = lookup_shipment(shipments, "PF-1003")
prompt = build_prompt("Where is my parcel PF-1003?", shipment, style_guide)
draft = complete(prompt, model="mock")
print("=" * 60)
print("DRAFT:")
print("=" * 60)
print(draft)

### Step 5 — OUTPUT the draft

The last piece. We could:
- Print it to the terminal (today — that's what a CLI does)
- Return it as JSON (so other programs can use it)
- Send it to an HTTP endpoint (Phase 2!)
- Save it to a file (so a human can review it)

Today we do BOTH print and JSON, because we want to be able to use this drafter from a terminal AND pipe its output to other programs.

In [ ]:
# Step 5 — OUTPUT the draft
import json as _json

def output_draft(draft: str, as_json: bool = False) -> None:
    """Output the draft to the terminal. as_json=True wraps it in a dict."""
    if as_json:
        print(_json.dumps({"draft": draft}, indent=2, ensure_ascii=False))
    else:
        print("=" * 60)
        print("DRAFT:")
        print("=" * 60)
        print(draft)
        print("=" * 60)

# Test the full pipeline
def run_drafter(email_source: str, shipment_id: Optional[str] = None, as_json: bool = False) -> dict:
    """The full 5-piece pipeline. Returns a result dict (also printed)."""
    email_text = read_email(email_source)            # 1️⃣ READ
    if not shipment_id:
        shipment_id = extract_shipment_id(email_text) # 2️⃣ EXTRACT
    if not shipment_id:
        return {"error": "No shipment ID found"}
    shipment = lookup_shipment(shipments, shipment_id) # 3️⃣ LOOKUP
    if not shipment:
        return {"error": f"Shipment {shipment_id} not found"}
    prompt = build_prompt(email_text, shipment, style_guide)  # 4a DRAFT (build prompt)
    draft = complete(prompt)                                  # 4b DRAFT (call LLM)
    output_draft(draft, as_json=as_json)                      # 5️⃣ OUTPUT
    return {"shipment_id": shipment_id, "draft": draft}

# Run it for real
result = run_drafter("Where is my parcel PF-1003? I sent it last Tuesday.", as_json=True)

## 🏛️ FDE Lens — the 5-piece anatomy is UNIVERSAL

Every AI tool you'll ever build is just these 5 pieces. Different tools, different data, different LLMs — but the 5 pieces are always the same:

```
  ┌──────────────────────────────────────────────────────────┐
  │                                                          │
  │  📥 READ     Get the input (email, image, voice, etc.)  │
  │       │                                                  │
  │       ▼                                                  │
  │  🔍 EXTRACT Pull out the key signal (ID, name, intent)  │
  │       │                                                  │
  │       ▼                                                  │
  │  📚 LOOKUP  Find the facts (database, API, file)        │
  │       │                                                  │
  │       ▼                                                  │
  │  ✍️  DRAFT    Ask the LLM to write the answer            │
  │       │                                                  │
  │       ▼                                                  │
  │  📤 OUTPUT   Deliver the result (print, HTTP, file)      │
  │                                                          │
  └──────────────────────────────────────────────────────────┘
```

**Three tools built with the same 5 pieces:**

| Tool | READ | EXTRACT | LOOKUP | DRAFT | OUTPUT |
|---|---|---|---|---|---|
| **Email drafter** (today) | email text | PF-1003 | shipments.json | reply | print |
| **Image describer** | photo file | dog, park | image embeddings | caption | JSON |
| **Voice transcriber** | audio file | "parcel?" | none | transcript | text |

When you start a new AI tool, **always draw the 5 boxes first.** Then fill them in. The boxes don't change.

## 🧪 Test your drafter

The drafter works on the simple case. Now let's test the edge cases — the places where it might fail. **A tool that only works on the happy path is a tool that breaks in production.**

Edge cases to test:
1. Email mentions the ID, but it's lowercase (`pf-1003`)
2. Email mentions TWO shipment IDs
3. Email mentions no shipment ID at all
4. Shipment ID is mentioned but doesn't exist in the database
5. Email is in Vietnamese (multi-language)

In [ ]:
# Edge case tests
edge_cases = [
    ("lowercase ID",  "where is pf-1003?"),
    ("two IDs",       "PF-1003 arrived but PF-1007 is missing"),
    ("no ID",         "do you ship to vietnam?"),
    ("unknown ID",    "where is PF-9999?"),
    ("vietnamese",    "cho tôi hỏi về PF-1003"),
]

for name, email in edge_cases:
    print(f"\n--- {name} ---")
    print(f"Email: {email!r}")
    found = extract_shipment_id(email)
    print(f"Regex found: {found!r}")
    if found:
        s = lookup_shipment(shipments, found)
        print(f"Lookup: {'OK' if s else 'NOT FOUND'}")
    print()

**What you learned from the edge cases:**

- **Lowercase ID:** our regex misses it (case-sensitive). Fix: pass `re.IGNORECASE`.
- **Two IDs:** our regex picks the FIRST. Fix: ask the user which one, or pick by recency.
- **No ID:** the drafter returns an error. Fix: ask the LLM to look up by customer email/name.
- **Unknown ID:** the drafter returns "not found." Fix: include a "did you mean?" search.
- **Vietnamese:** the regex still finds PF-1003. The LLM will probably reply in English (Phase 2 adds translation).

**The lesson:** every tool has edge cases. The FDE's job is to find them BEFORE the customer does, and either fix them or document them.

## 🌙 Reflect

**Write down 3 things before you move on:**

1. The 5 pieces of the drafter (READ, EXTRACT, LOOKUP, DRAFT, OUTPUT). Without looking.
2. Why we use a regex to find the ID before asking the LLM. (Hint: speed, cost, accuracy.)
3. One edge case the drafter doesn't handle well, and how you'd fix it.

**If you can't write these down, re-read the notebook. The 5-piece anatomy is the foundation of every other lesson in this course.**

## 🎓 What's next

You have a working drafter. But:

- It only runs in your terminal (Mei has to be at her laptop)
- It doesn't remember anything (no chat history)
- It can't find shipments by customer name (only by ID)
- It has no tests (you can't tell if it broke)

**Phase 2 fixes all of these.** The drafter becomes a service that anyone at PacificFreight can call over HTTP. It remembers context. It searches by customer name. It has a test suite. And it has 4 new endpoints (`/draft`, `/retrieve`, `/eval`, `/health`).

Go to [Phase 2 — Core Build](../../phase-2-core-build/) to see how the drafter grows from a seed into a service.